# Experimentations
This notebook permits to run experimentations to MLFlow locally

In [1]:
import mlflow
import mlflow.lightgbm
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import pandas as pd
import os
from mlflow.tracking import MlflowClient
os.environ["GIT_PYTHON_REFRESH"] = "quiet"

# check tracking URI
assert "MLFLOW_TRACKING_URI" in os.environ, "MLFLOW_TRACKING_URI non définie"

experiment_name = "1-lightgbm"

client = MlflowClient()

exp = client.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(experiment_name)
else:
    exp_id = exp.experiment_id

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, make_scorer, precision_recall_curve
from lightgbm import LGBMClassifier
import mlflow
import mlflow.lightgbm
import matplotlib.pyplot as plt

# Load prepared data
df = pd.read_csv("../.data/modelisation/app_train.csv")
target_col = "TARGET"
X = df.drop(columns=[target_col])
y = df[target_col]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Prepare scoring
scoring = {
    "auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

# TRÈS SIMPLE : seuil pour maximiser le rappel
def find_optimal_recall_threshold(y_true, y_proba):
    _, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    # rappel max atteint au seuil le plus bas (souvent thresholds[0])
    max_recall_idx = np.argmax(recalls[1:])  # on ignore le premier point (seuil=1)
    return thresholds[max_recall_idx]

# run MLflow
with mlflow.start_run(experiment_id=exp_id):
    mlflow.log_artifact("2-experimentations.ipynb", artifact_path="notebooks")

    # Instantiate model with cost function (10x coeff)
    model = LGBMClassifier(n_estimators=100, random_state=42, class_weight={0: 1, 1: 10})

    # Cross-validation on training set
    cv_results = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=5,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    # Log CV metrics (mean and std)
    for metric_name in scoring.keys():
        test_key = f"test_{metric_name}"
        mean_val = round(float(np.mean(cv_results[test_key])), 2)
        std_val = float(np.std(cv_results[test_key]))
        mlflow.log_metric(f"{metric_name}", mean_val)
        #mlflow.log_metric(f"{metric_name}_std", std_val)

    # Log params
    mlflow.log_params(model.get_params())

    # Fit final model on the whole training set (to evaluate on X_test)
    model.fit(X_train, y_train)
    
    # Predict probabilities on test set
    y_proba = model.predict_proba(X_test)[:, 1]

    # seuil pour maximiser le rappel
    recall_threshold = find_optimal_recall_threshold(y_test, y_proba)
    y_pred_recall = (y_proba >= recall_threshold).astype(int)
    recall_score_max = recall_score(y_test, y_pred_recall)
    precision_score_max = precision_score(y_test, y_pred_recall)
    
    # Log recall threshold and metrics
    mlflow.log_metric("optimal_recall_threshold", round(recall_threshold, 3))
    mlflow.log_metric("recall_at_optimal", round(recall_score_max, 3))
    mlflow.log_metric("precision_at_optimal_recall", round(precision_score_max, 3))

    # Log model
    try:
        mlflow.lightgbm.log_model(model.booster_, name="model")
    except TypeError:
        # fallback if signature changed
        mlflow.lightgbm.log_model(model.booster_, name="model")

[LightGBM] [Info] Number of positive: 19876, number of negative: 226132
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026799 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12543
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 261
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.467789 -> initscore=-0.129021
[LightGBM] [Info] Start training from score -0.129021


2025/10/14 08:59:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run unleashed-snake-776 at: http://mlflow:5000/#/experiments/696573625829308745/runs/024b7d25abc64211877d6a3f19423e8e
🧪 View experiment at: http://mlflow:5000/#/experiments/696573625829308745
